# 🔬 ABPT Research Campaign
**Anchor-Based Probing Tool — Autonomous Research Loop**

Этот ноутбук запускает автономную исследовательскую кампанию:
1. Клонирует репо
2. Устанавливает зависимости
3. Запускает Orchestrator (Strategist → Worker → Analyzer)
4. Обновляет playbook.md и research_state.json

**Архитектура:** AlphaLab-style persistent playbook + Karpathy-style greedy loop

---
⚡ Runtime: `GPU → T4 / A100`  
⏱ Время: ~30–60 мин на эксперимент

## 0. GPU Check

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️ GPU не обнаружен — смени Runtime: Runtime → Change runtime type → GPU')

## 1. Клонирование репо

In [ ]:
import os

REPO_URL = 'https://github.com/YOUR_USERNAME/ABPT.git'  # ← замени на свой URL
REPO_DIR = '/content/ABPT'

if os.path.exists(REPO_DIR):
    print('Репо уже есть → git pull')
    !cd {REPO_DIR} && git pull
else:
    print('Клонирую...')
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Рабочая директория: {os.getcwd()}')
!git log --oneline -5

### 1b. Альтернатива: загрузить архив вручную
Если нет публичного репо — загрузи `ABPT_full_repository.tar.gz` и распакуй:

In [ ]:
# АЛЬТЕРНАТИВА: загрузить zip/tar вручную
# from google.colab import files
# uploaded = files.upload()  # загрузи ABPT.tar.gz
# !tar -xzf ABPT.tar.gz -C /content/
# os.chdir('/content/ABPT')
print('(ячейка пропущена — используем git clone выше)')

## 2. Установка зависимостей

In [ ]:
%%time
!pip install -q \
    transformers==4.47.0 \
    accelerate \
    scipy \
    plotly \
    scikit-learn \
    watchdog
print('✅ Зависимости установлены')

## 3. Загрузка модели (кеш)

In [ ]:
%%time
# Предзагрузка модели чтобы не скачивать при каждом эксперименте
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = 'Qwen/Qwen3.5-4B'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Загружаю {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
print(f'✅ Модель загружена на {device}')
print(f'   Параметров: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B')

## 4. Текущий State кампании

In [ ]:
import json
from pathlib import Path

state = json.loads(Path('research_state.json').read_text())

print(f"📊 ABPT Research Campaign")
print(f"   Модель:  {state['model']}")
print(f"   Фаза:    {state['current_phase']}/6")
print(f"   Бюджет:  {state['budget_remaining']}/{state['experiments_max']} экспериментов")
print(f"   Запущено: {state.get('experiments_run', 0)}")
print()
print("Открытые гипотезы:")
for h in state.get('open_hypotheses', []):
    status_icon = {'untested': '⬜', 'success': '✅', 'failed': '❌', 'skipped': '⏭️'}.get(h['status'], '?')
    print(f"  {status_icon} {h['id']} (Фаза {h['phase']}): {h['statement'][:60]}...")

print()
print("Метрики:")
for m in state.get('metric_history', [])[-5:]:
    print(f"  {m['experiment']}: {m['metric']} = {m['value']}")

## 5. 🚀 Запуск Orchestrator

In [ ]:
# Сколько экспериментов запустить в этой сессии
BUDGET_THIS_SESSION = 3

# Опционально: API ключ для LLM-assisted Strategist
# import os; os.environ['OPENAI_API_KEY'] = 'sk-...'

!python scripts/orchestrate.py --budget {BUDGET_THIS_SESSION}

## 5b. Запустить конкретную гипотезу

In [ ]:
# Запустить только H1 — phase probe (Фаза 1)
!python scripts/orchestrate.py --experiment H1

## 5c. Только phase probe без orchestrator

In [ ]:
%%time
# Прямой запуск phase probe (без orchestrator)
!python scripts/run_qwen_phase_probe.py \
    --model Qwen/Qwen3.5-4B \
    --anchor-profile medium \
    --tau 0.5

## 6. Результаты

In [ ]:
import json
from pathlib import Path

# Показать последний phase probe результат
archive = Path('archive')
probes = sorted(archive.glob('*phase_probe*.json'), key=lambda p: p.stat().st_mtime, reverse=True)

if not probes:
    print('Нет результатов phase probe. Запусти ячейку 5b или 5c.')
else:
    data = json.loads(probes[0].read_text())
    print(f'📄 Файл: {probes[0].name}')
    print(f'   Кейсов: {data["metadata"]["n_cases"]}')
    print(f'   Профиль: {data["metadata"]["anchor_profile"]}')
    print()
    
    corr = data['correlation_summary']
    print('🔗 Корреляции (Spearman ρ vs base_constraint_score):')
    main_rho = corr.get('spearman_early_slope_4_8_vs_base_constraint')
    print(f'   early_slope_4_8: ρ = {main_rho:.4f}' if main_rho else '   early_slope_4_8: N/A')
    
    for metric, rho in corr.get('all_metrics', {}).items():
        if rho is not None:
            bar = '█' * int(abs(rho) * 20)
            sign = '+' if rho >= 0 else '-'
            print(f'   {metric:<30} {sign}{bar} {rho:.3f}')
    
    print()
    print('📋 Per-case:')
    print(f'{"name":<35} {"slope":>8} {"peak_L":>7} {"sharpness":>10} {"constraint":>12}')
    print('-' * 75)
    for c in data['cases']:
        pm = c['phase_metrics']
        slope = pm.get('early_slope_4_8')
        peak_l = pm.get('peak_layer_4_12')
        sharp = pm.get('sharpness')
        cs = c['base_constraint_score']
        slope_s = f'{slope:.4f}' if slope is not None else '—'
        peak_s = f'L{peak_l}' if peak_l is not None else '—'
        sharp_s = f'{sharp:.3f}' if sharp is not None else '—'
        print(f'{c["name"]:<35} {slope_s:>8} {peak_s:>7} {sharp_s:>10} {cs:>12.0f}')

## 7. Визуализация

In [ ]:
import json
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

# Загружаем последний phase probe
probes = sorted(Path('archive').glob('*phase_probe*.json'), key=lambda p: p.stat().st_mtime, reverse=True)
if not probes:
    print('Нет данных. Сначала запусти phase probe.')
else:
    data = json.loads(probes[0].read_text())
    cases = data['cases']
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            'early_slope_4_8 vs base_constraint',
            'r1 profiles по слоям',
            'sharpness vs constraint',
            'Все корреляции',
        ]
    )
    
    # 1. Scatter: slope vs constraint
    slopes = [c['phase_metrics'].get('early_slope_4_8') for c in cases]
    constr = [c['base_constraint_score'] for c in cases]
    names = [c['name'].replace('_', ' ')[:20] for c in cases]
    colors = ['#2ecc71' if cs == 1.0 else '#e74c3c' for cs in constr]
    
    fig.add_trace(go.Scatter(
        x=slopes, y=constr, mode='markers+text',
        text=names, textposition='top center',
        marker=dict(color=colors, size=10),
        name='cases',
    ), row=1, col=1)
    
    # 2. r1 profiles
    for c in cases:
        r1 = c['r1_profile']
        layers = sorted([int(k) for k in r1.keys()])
        vals = [r1[str(l)] for l in layers]
        cs = c['base_constraint_score']
        color = '#2ecc71' if cs == 1.0 else '#e74c3c'
        fig.add_trace(go.Scatter(
            x=layers, y=vals, mode='lines',
            line=dict(color=color, width=1.5),
            name=c['name'][:15],
            showlegend=False,
        ), row=1, col=2)
    
    # Зона кристаллизации
    fig.add_vrect(x0=4, x1=8, fillcolor='rgba(255,255,0,0.1)',
                  line_width=0, row=1, col=2)
    
    # 3. Sharpness vs constraint
    sharps = [c['phase_metrics'].get('sharpness') for c in cases]
    fig.add_trace(go.Scatter(
        x=sharps, y=constr, mode='markers',
        marker=dict(color=colors, size=10),
        name='sharpness',
    ), row=2, col=1)
    
    # 4. Bar chart корреляций
    corr_data = data['correlation_summary']['all_metrics']
    m_names = list(corr_data.keys())
    m_vals = [corr_data[m] or 0 for m in m_names]
    bar_colors = ['#2ecc71' if v > 0.4 else '#e74c3c' if v < -0.4 else '#95a5a6' for v in m_vals]
    
    fig.add_trace(go.Bar(
        x=[m.replace('_', ' ')[:20] for m in m_names],
        y=m_vals,
        marker_color=bar_colors,
        name='ρ',
    ), row=2, col=2)
    
    fig.add_hline(y=0.4, line_dash='dash', line_color='yellow', row=2, col=2)
    fig.add_hline(y=-0.4, line_dash='dash', line_color='yellow', row=2, col=2)
    
    fig.update_layout(
        title=f'ABPT Phase Probe Results — {probes[0].name}',
        template='plotly_dark',
        height=700,
        showlegend=False,
    )
    fig.show()

## 8. Обновлённый State и Playbook

In [ ]:
# Показать финальный state
!python scripts/orchestrate.py --status

print('\n--- PLAYBOOK (последние 50 строк) ---')
!tail -50 playbook.md

## 9. Сохранить результаты (download)

In [ ]:
import shutil
from google.colab import files
from pathlib import Path
from datetime import datetime

# Упаковываем archive/ + playbook.md + research_state.json
timestamp = datetime.now().strftime('%Y%m%d_%H%M')
output_name = f'abpt_campaign_{timestamp}'

import os
os.makedirs(f'/tmp/{output_name}', exist_ok=True)
shutil.copytree('archive', f'/tmp/{output_name}/archive', dirs_exist_ok=True)
shutil.copy('playbook.md', f'/tmp/{output_name}/')
shutil.copy('research_state.json', f'/tmp/{output_name}/')
if Path('orchestrator_log.jsonl').exists():
    shutil.copy('orchestrator_log.jsonl', f'/tmp/{output_name}/')

shutil.make_archive(f'/tmp/{output_name}', 'zip', '/tmp', output_name)
files.download(f'/tmp/{output_name}.zip')
print(f'✅ Скачан: {output_name}.zip')

---
## 📋 Шпаргалка

| Команда | Что делает |
|---------|------------|
| `!python scripts/orchestrate.py` | Запустить следующий эксперимент |
| `!python scripts/orchestrate.py --budget 5` | Запустить 5 экспериментов подряд |
| `!python scripts/orchestrate.py --experiment H1` | Запустить конкретную гипотезу |
| `!python scripts/orchestrate.py --dry-run` | Показать план без запуска |
| `!python scripts/orchestrate.py --status` | Текущий state |
| `!python scripts/run_qwen_phase_probe.py --anchor-profile medium` | Phase probe напрямую |

**Гипотезы:**
- `H1` — early_slope_4_8 vs base_constraint_score (Фаза 1, main)
- `H1_short` / `H1_long` — то же для других профилей
- `H4` — attention beacon verification
- `H2` — group-specific routing (Фаза 2, требует H1)
- `H3` — injection detection (Фаза 3, требует H1)

**После сессии:** скачай ячейкой 9 и распакуй в `Desktop/ABPT/archive/`